# kld_sweep.py — Linux / Colab test
Tests `kld_sweep.py` on **Qwen3.5-0.8B** using two small quants from unsloth.
- BF16: 1.52 GiB
- Quants: UD-Q4_K_XL (559 MB) + UD-IQ2_XXS (338 MB)
- Total disk: ~2.5 GiB

**Runtime:** GPU (T4 recommended). CPU-only will work but logits generation will be slow.

**Steps:** Run all cells in order. The sweep resumes automatically if interrupted.

In [ ]:
# @title 1 — Install dependencies
!pip install pandas matplotlib adjustText scipy -q

In [ ]:
# @title 2 — Build llama.cpp from source (CUDA, T4 optimized)
# No prebuilt Linux CUDA binary available — build takes ~3-5 minutes
import os

!apt-get install -y cmake libcurl4-openssl-dev -q
!git clone https://github.com/ggml-org/llama.cpp --depth 1 -q
!cmake llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_BUILD_TYPE=Release -q
!cmake --build llama.cpp/build --target llama-perplexity -j$(nproc)

PPL_EXE = 'llama.cpp/build/bin/llama-perplexity'
assert os.path.exists(PPL_EXE), f'Build failed — {PPL_EXE} not found'
print(f'\nllama-perplexity: {PPL_EXE}')


In [ ]:
# @title 3 — Download BF16 model and quants from HuggingFace
!pip install huggingface_hub -q
from huggingface_hub import hf_hub_download
import os

os.makedirs('bf16',   exist_ok=True)
os.makedirs('quants', exist_ok=True)

REPO = 'unsloth/Qwen3.5-0.8B-GGUF'

bf16_path = hf_hub_download(REPO, 'Qwen3.5-0.8B-BF16.gguf',     local_dir='bf16')
q1_path   = hf_hub_download(REPO, 'Qwen3.5-0.8B-UD-Q4_K_XL.gguf', local_dir='quants')
q2_path   = hf_hub_download(REPO, 'Qwen3.5-0.8B-UD-IQ2_XXS.gguf', local_dir='quants')

print(f'BF16  : {bf16_path}')
print(f'Quant1: {q1_path}')
print(f'Quant2: {q2_path}')

In [ ]:
# @title 4 — Download kld_sweep.py and dataset
import os

# Fetch script from GitHub
!wget -q https://raw.githubusercontent.com/cmhamiche/kld-sweep/main/kld_sweep.py
assert os.path.exists('kld_sweep.py'), 'Failed to download kld_sweep.py'
print('kld_sweep.py downloaded')

# Dataset — wikitext2 test set via llama.cpp helper
!wget -q https://raw.githubusercontent.com/ggml-org/llama.cpp/master/scripts/get-wikitext-2.sh
!bash get-wikitext-2.sh
DATASET = 'wikitext-2-raw/wiki.test.raw'
assert os.path.exists(DATASET), f'Dataset not found: {DATASET}'
print(f'Dataset: {DATASET}')
SCRIPT = 'kld_sweep.py'


In [ ]:
# @title 5 — Run sweep
import subprocess, sys

cmd = [
    sys.executable, SCRIPT,
    '--exe',        PPL_EXE,
    '--bf16',       bf16_path,
    '--quants',     'quants',
    '--dataset',    DATASET,
    '--output',     'output',
    '--model-name', 'Qwen3.5-0.8B',
    '--args',       '-t 4 -c 512 -ngl 99',
]

print('Command:', ' '.join(cmd))
result = subprocess.run(cmd)
print('\nExit code:', result.returncode)

In [ ]:
# @title 6 — Show results
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('output/Qwen3.5-0.8B_results.csv')
display(df.sort_values('KLD_Score'))

display(Image('output/kld_plot_Qwen3.5-0.8B.png'))
display(Image('output/ppl_plot_Qwen3.5-0.8B.png'))